# 原Stage1组成映射失败区间归因

固定范围见`plan.json`。只读封存Qlib Recorder的一个0bp候选账户、原生估值点、转换记录、全部季度及连续窗口，描述失败发生在哪里，不搜索交易阈值。

In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import qlib
from src.core.recorder_sources import load_verified_recorder_objects

root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root: raise RuntimeError("workspace missing")
    root=root.parent
card=root/"research/stage1_beta_composition_attribution_20261022"
plan=json.loads((card/"plan.json").read_text())
source=root/plan["source_card"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
for file,key in (("summary.json","source_summary_sha256"),("verification_manifest.json","source_verification_manifest_sha256"),("quarter_metrics.csv","source_quarter_metrics_sha256"),("readback_verification.json","source_readback_receipt_sha256")):
    assert sha(source/file)==plan[key],file
summary=json.loads((source/"summary.json").read_text())
assert summary["decision"]=="not_promoted_fixed_Stage1_beta_composition_0bp"
assert summary["recorder_id"]==plan["source_recorder_id"] and summary["new_native_accounts"]==1
qlib.init(provider_uri=str(root/"data/qlib/stage1_domain_heads_20261018"),region="us",kernels=1)
objects,readback=load_verified_recorder_objects(source,expected_recorder_id=plan["source_recorder_id"],artifacts=plan["artifacts"],expected_manifest_sha256=plan["source_verification_manifest_sha256"])
book=objects["native/candidate_0"];path=objects["paths/candidate_0"]
rolling=objects["analysis/rolling/all_windows"];audit=objects["analysis/composition_audit"]
assert book["audit"]["status"]=="passed" and len(book["authority"])==len(path["replay"])==944
assert len(rolling)==819 and len(audit)==summary["composition_audit"]["transform_calls"]==919
points=book["points"].copy();stamps=pd.to_datetime(points.stamp);nav=points.nav.to_numpy(float)
peak=np.maximum.accumulate(np.r_[100000.,nav])[1:]
drawdown=nav/peak-1
trough=int(np.argmin(drawdown));prior=np.r_[100000.,nav[:trough+1]];peak_at=int(np.argmax(prior))
peak_stamp="initial_cash" if peak_at==0 else str(stamps.iloc[peak_at-1])
breaches=np.flatnonzero(drawdown<-0.30)
account=path["replay"].copy();targets=path["targets"].copy()
assert pd.DatetimeIndex(account.trade_date).equals(pd.DatetimeIndex(targets.trade_date))
qmetrics=pd.read_csv(source/"quarter_metrics.csv")
periods=pd.DatetimeIndex(account.trade_date).to_period("Q").astype(str)
rows=[]
for period in pd.unique(periods):
    mask=np.asarray(periods==period)
    held=account.loc[mask];intended=targets.loc[mask]
    metric=qmetrics.loc[qmetrics.period.eq(period)]
    assert len(metric)==1 and len(held)==int(metric.iloc[0].sessions)
    rows.append({"period":period,"sessions":len(held),"complete_period":bool(metric.iloc[0].complete_period),
        "SOXX_relative_wealth":float(metric.iloc[0].relative_wealth),
        "target_soxl_mean":float(intended.soxl_weight.mean()),"target_cash_mean":float(intended.cash_weight.mean()),
        "ending_soxl_mean":float(held.ending_soxl_weight.mean()),"ending_cash_mean":float(held.ending_cash_weight.mean()),
        "risk_state_ge2_fraction":float(held.risk_state.ge(2).mean()),
        "open_execute_fraction":float(held.execute_open_target.mean()),
        "direct_fees_usd":float(held.broker_and_external_fees_usd.sum())})
quarter=pd.DataFrame(rows)
assert len(quarter)==16 and quarter.sessions.sum()==944
quarter.to_csv(card/"quarterly_path_attribution.csv",index=False)
worst=rolling.loc[rolling.relative_log.idxmin()]
result={"status":"descriptive_fixed_failure_attribution_no_parameter_search",
    "source_recorder_id":plan["source_recorder_id"],"source_plan_sha256":summary["plan_sha256"],
    "plan_sha256":sha(card/"plan.json"),"native_sessions":944,"quarters":16,
    "rolling_windows":819,"positive_rolling_windows":int(rolling.relative_log.gt(0).sum()),
    "worst_126_window":{"first_trade":str(worst.first_trade),"last_exit":str(worst.last_exit),
                         "relative_return_pct":float(worst.relative_return_pct)},
    "max_observed_drawdown_pct":float(100*drawdown[trough]),
    "max_drawdown_peak_stamp":peak_stamp,"max_drawdown_trough_stamp":str(stamps.iloc[trough]),
    "observed_marks_below_minus30pct":int(len(breaches)),
    "first_breach_stamp":str(stamps.iloc[breaches[0]]) if len(breaches) else None,
    "negative_complete_quarters":[r["period"] for r in rows if r["complete_period"] and r["SOXX_relative_wealth"]<1],
    "risk_and_exposure_examples":{name:next(r for r in rows if r["period"]==name)
                                 for name in ("2024Q3","2026Q2","2026Q3")},
    "total_direct_fees_usd":float(account.broker_and_external_fees_usd.sum()),
    "transformation_calls":len(audit),"recovery_transformation_calls":int(audit.slot.eq("recovery").sum()),
    "causal_effect_identified":False,"new_fits":0,"new_accounts":0,"new_bootstraps":0,
    "decision":"original_fixed_0bp_version_remains_not_promoted"}
(card/"failure_attribution.json").write_text(json.dumps(result,indent=2)+"\n")
side={"experiment":plan["experiment"],"decision":result["decision"],"objective_achieved":False,
      "source_recorder_id":plan["source_recorder_id"],"source_native_qlib_readback":True,
      "new_fits":0,"new_native_accounts":0,"new_bootstraps":0,
      "artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml",
                         "recorder_only_artifacts":[],"source_recorder_authority":True}}
(card/"summary.json").write_text(json.dumps(side,indent=2)+"\n")
qgood=result["risk_and_exposure_examples"]["2024Q3"]
qbad=result["risk_and_exposure_examples"]["2026Q3"]
conclusion=f"""# 原Stage1固定组成0bp：一次失败区间归因

只读原生Qlib Recorder，保留944个账户区间、16个季度和819个连续126日窗口；0新拟合、0新账户、0新bootstrap。固定账户累计相对SOXX财富1.077455，但可用点最大回撤{result['max_observed_drawdown_pct']:.4f}%，峰值{peak_stamp}、低点{result['max_drawdown_trough_stamp']}，共有{len(breaches)}个观测点越过30%。最差126日窗口{result['worst_126_window']['first_trade']}至{result['worst_126_window']['last_exit']}，相对SOXX收益{result['worst_126_window']['relative_return_pct']:.2f}%；正超额{result['positive_rolling_windows']}/819，未到70%。

2024Q3相对SOXX财富{qgood['SOXX_relative_wealth']:.4f}、平均期末现金{qgood['ending_cash_mean']:.1%}；2026Q3相对财富{qbad['SOXX_relative_wealth']:.4f}、平均期末现金{qbad['ending_cash_mean']:.1%}，风险状态>=2占{qbad['risk_state_ge2_fraction']:.1%}。两季及所有其他季度的目标/实际SOXL、现金、执行比例和直接费用均在`quarterly_path_attribution.csv`。这是同一路径的区间定位，不能把坏季或状态直接变成可交易开关，也不能把费用/风控状态视为独立因果效应。

决定：`original_fixed_0bp_version_remains_not_promoted`。本固定映射虽然提高全程终值，但0bp回撤、滚动稳定性与区块下界失败；不能按复用历史调整映射比例、阈值或窗口。后续需要新决策时信息来区分在弱势中应保留防守与恢复时应提高参与，先过独立信息/来源门，然后在完整原Stage1链检验。
"""
(card/"conclusion.md").write_text(conclusion)
print(json.dumps({"decision":result["decision"],"max_drawdown_pct":result["max_observed_drawdown_pct"],"worst_window_pct":result["worst_126_window"]["relative_return_pct"],"new_accounts":0}))

## 中文结论

执行后以`conclusion.md`、`failure_attribution.json`和`quarterly_path_attribution.csv`为准。来源Recorder保持不变，负结论不因归因而重开。